In [ ]:
import os
import sys

# Add project root to path
nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)

In [ ]:
import numpy as np
import gymnasium as gym
import gymnasium_env_2  # Register environments

## Available Environment Variants

**Observation Spaces:**
- `LDPC-FullLLR-v0`: Full LLR vector (972 dimensions)
- `LDPC-LLRStats-v0`: Statistical features (7 dimensions)
- `LDPC-Residuals-v0`: Residuals per cluster (8 dimensions)
- `LDPC-SyndromeHistory-v0`: Syndrome weight history (7 dimensions)

**Reward Functions:**
- `LDPC-SyndromeReward-v0`: Reward for reducing syndrome weight
- `LDPC-SparseReward-v0`: Reward only at episode end
- `LDPC-TimeEfficiency-v0`: Reward for fast convergence
- `LDPC-ResidualReward-v0`: Reward for scheduling high-residual clusters
- `LDPC-BalancedScheduling-v0`: Reward for balanced cluster usage

## Test 1: Full LLR Observation Space

In [ ]:
env = gym.make("gymnasium_env_2/LDPC-FullLLR-v0", num_clusters=6, max_iterations=30, snr_db=0)

print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")
print(f"Observation shape: {env.observation_space.shape}")

obs, info = env.reset(seed=42)
print(f"\nInitial observation stats:")
print(f"  Shape: {obs.shape}")
print(f"  Mean:  {np.mean(obs):.4f}")
print(f"  Std:   {np.std(obs):.4f}")

In [ ]:
# Run a few steps
for step in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {step+1}: action={action}, reward={reward:.1f}, syndrome={info['syndrome_weight']}")
    
    if terminated:
        print(f"Converged! Correct: {info['decoded_correctly']}")
        break

## Test 2: LLR Statistics Observation Space (Compact)

In [ ]:
env_stats = gym.make("gymnasium_env_2/LDPC-LLRStats-v0", num_clusters=6, max_iterations=30, snr_db=0)

print(f"Observation space: {env_stats.observation_space}")
print(f"Observation shape: {env_stats.observation_space.shape}")

obs, info = env_stats.reset(seed=42)
print(f"\nInitial observation (7 features):")
print(f"  {obs}")
print(f"\nFeatures: [mean, std, min, max, median, syndrome_weight, iteration]")

In [ ]:
# Run a few steps
for step in range(5):
    action = 0  # Always choose action 0
    obs, reward, terminated, truncated, info = env_stats.step(action)
    
    print(f"Step {step+1}: {obs}")
    
    if terminated:
        print(f"Converged! Correct: {info['decoded_correctly']}")
        break

## Test 3: Residuals Observation Space

In [ ]:
env_residuals = gym.make("gymnasium_env_2/LDPC-Residuals-v0", num_clusters=6, max_iterations=30, snr_db=0)

print(f"Observation space: {env_residuals.observation_space}")
print(f"Observation shape: {env_residuals.observation_space.shape}")

obs, info = env_residuals.reset(seed=42)
print(f"\nInitial observation (8 features):")
print(f"  Cluster residuals (6): {obs[:6]}")
print(f"  Syndrome weight:       {obs[6]}")
print(f"  Iteration progress:    {obs[7]}")

In [ ]:
# Run a few steps and watch residuals
for step in range(5):
    action = np.argmax(obs[:6])  # Choose cluster with highest residual
    obs, reward, terminated, truncated, info = env_residuals.step(action)
    
    print(f"Step {step+1}: action={action}, cluster_residuals={obs[:6]}")
    
    if terminated:
        print(f"Converged! Correct: {info['decoded_correctly']}")
        break

## Test 4: Different Reward Functions

In [ ]:
# Compare rewards from different environments
env_ids = [
    "gymnasium_env_2/LDPC-FullLLR-v0",
    "gymnasium_env_2/LDPC-SyndromeReward-v0",
    "gymnasium_env_2/LDPC-TimeEfficiency-v0",
]

print("Comparing rewards for same action sequence:\n")

for env_id in env_ids:
    env = gym.make(env_id, num_clusters=6, max_iterations=30, snr_db=0)
    obs, info = env.reset(seed=42)
    
    total_reward = 0
    for step in range(5):
        action = 0  # Same action
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
    
    print(f"{env_id.split('/')[-1]:25s}: Total reward = {total_reward:8.2f}")
    env.close()

## Test 5: Full Episode Comparison

In [ ]:
def run_episode(env_id, seed=42, max_steps=30):
    """Run one episode with random policy"""
    env = gym.make(env_id, num_clusters=6, max_iterations=max_steps, snr_db=0)
    obs, info = env.reset(seed=seed)
    
    total_reward = 0
    for step in range(max_steps):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        
        if terminated:
            break
    
    result = {
        'steps': info['iteration'],
        'total_reward': total_reward,
        'converged': info['is_converged'],
        'correct': info['decoded_correctly'],
    }
    
    env.close()
    return result

# Test different variants
variants = [
    "gymnasium_env_2/LDPC-FullLLR-v0",
    "gymnasium_env_2/LDPC-LLRStats-v0",
    "gymnasium_env_2/LDPC-Residuals-v0",
]

print("Random policy performance (single episode, seed=42):\n")
for variant in variants:
    result = run_episode(variant, seed=42)
    print(f"{variant.split('/')[-1]:25s}: Steps={result['steps']:2d}, "
          f"Reward={result['total_reward']:6.2f}, Correct={result['correct']}")

## Summary

You now have a modular framework where you can:

1. **Easily create new observation spaces** - inherit from `LDPCBaseEnv` and implement `_create_observation_space()` and `_get_obs()`
2. **Easily create new reward functions** - inherit from any observation variant and implement `_compute_reward()`
3. **Mix and match** - combine different observations with different rewards
4. **Test systematically** - use the comparison script to evaluate all variants

Next steps:
- Train RL agents on different variants
- Compare learning curves
- Find the best observation/reward combination for your use case